# Does Node2Vec already know how busy a road's surroundings are?

**Why this check.** Adding the OpenStreetMap (OSM) features raised GraphSAGE's test AUROC by 0.0252 without Node2Vec but only by 0.0029 with it.
The model relies mostly on the *nearby places* features (counts of signals, food outlets, crossings and so on around each road).
If the Node2Vec embedding already encodes how built-up a road's surroundings are, that would explain why the OSM features add little once Node2Vec is present.

**What it does.** For every physical road, predict its nearby-place counts from its Node2Vec vector alone, and measure how well that works on parts of
Delaware the model has not seen (spatial blocks), and then on roads drawn at random (section 6), whose areas the model has seen.
High R² on unseen areas would mean Node2Vec encodes surroundings in general; high R² only on seen areas means it identifies *where* a road is.

**Needs (in Google Drive):** `DE_osm_features.parquet` and `DE.zip` in `MyDrive/cs5284_project/data/`, and the Node2Vec file `DE_128.npy` somewhere under
`MyDrive/cs5284_project/` (it is searched for). Nothing is trained on accidents; no GPU is needed. Runtime: a few minutes.

**Output:** printed tables and `node2vec_density_check_DE.json` in `MyDrive/cs5284_project/results/`.

## 1. Setup and inputs

In [1]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/cs5284_project")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
STATE = "DE"

FEATURE_TABLE_PATH = DATA_DIR / f"{STATE}_osm_features.parquet"
BENCHMARK_ZIP_PATH = DATA_DIR / f"{STATE}.zip"

PLACE_KINDS = ["signal", "food", "crossing", "parking", "bus_stop", "school", "bar_pub"]
RADII_M = [250, 1000]
GRID_SIZE = 10          # spatial blocks: GRID_SIZE x GRID_SIZE cells over the state
NUMBER_OF_FOLDS = 5
RANDOM_SEED = 0
RIDGE_ALPHA = 1.0

Mounted at /content/drive


In [2]:
def find_node2vec_file(search_root, state, dimensions=128):
    """Find the cached Node2Vec embedding for a state.

    Why: the comparison notebook cached it on Drive, but the exact sub-folder depends on how it was run.
    Takes: the folder to search, the state code and the embedding size. Returns the path; raises if none or several differ.
    """
    candidates = sorted(search_root.rglob(f"{state}_{dimensions}.npy"))
    if not candidates:
        raise FileNotFoundError(f"No {state}_{dimensions}.npy under {search_root}. Set NODE2VEC_PATH by hand.")
    for path in candidates:
        print("found:", path)
    return candidates[0]


def load_node_coordinates(benchmark_zip_path):
    """Read every intersection's latitude and longitude from the benchmark zip, in node-index order.

    Why: spatial blocks need positions. The release labels the two columns the wrong way round, so they are swapped back
    and checked against Delaware's bounding box. Takes the zip path. Returns an array [number of nodes, 2] of (lat, lon).
    """
    with zipfile.ZipFile(benchmark_zip_path) as archive:
        member = sorted(n for n in archive.namelist() if "/Nodes/node_features_" in "/" + n and n.endswith(".csv"))[0]
        with archive.open(member) as stream:
            nodes = pd.read_csv(stream, usecols=["lat", "lon"])
    nodes = nodes.rename(columns={"lat": "lon", "lon": "lat"})   # undo the swap in the release
    assert nodes["lat"].between(37, 41).all() and nodes["lon"].between(-80, -74).all(), "coordinate swap check failed"
    return nodes[["lat", "lon"]].to_numpy()


NODE2VEC_PATH = find_node2vec_file(PROJECT_DIR, STATE)
node_embeddings = np.load(NODE2VEC_PATH).astype(np.float64)
feature_table = pd.read_parquet(FEATURE_TABLE_PATH)
node_coordinates = load_node_coordinates(BENCHMARK_ZIP_PATH)

print("Node2Vec:", node_embeddings.shape, "| feature table:", feature_table.shape, "| nodes with coordinates:", len(node_coordinates))
assert len(node_embeddings) == len(node_coordinates), "embedding rows and node count differ"

found: /content/drive/MyDrive/cs5284_project/data/embeddings/node2vec_merged/DE_128.npy
Node2Vec: (49023, 128) | feature table: (116196, 76) | nodes with coordinates: 49023


## 2. One row per physical road

The feature table has one row per *directed* road; both directions of a road carry the same nearby-place counts. We keep one row per physical road.
A road's Node2Vec vector is the average of its two end intersections' vectors, and its position is the midpoint of its two ends.

In [3]:
def build_physical_road_table(feature_table, node_embeddings, node_coordinates, place_kinds, radii_m):
    """One row per physical road: its Node2Vec vector, its midpoint, and its nearby-place targets.

    Takes: the OSM feature table (directed rows with node_1_idx, node_2_idx), the node embeddings and coordinates, and which
    place kinds and radii to use as targets. Returns (road_vectors, road_midpoints, targets DataFrame).
    Targets are log(1 + count), the same transform the model uses; 'all_places_1000m' adds every kind within 1 km.
    """
    start = feature_table["node_1_idx"].to_numpy()
    end = feature_table["node_2_idx"].to_numpy()

    # Keep the first row of each unordered (start, end) pair.
    number_of_nodes = len(node_embeddings)
    pair_key = np.minimum(start, end).astype(np.int64) * number_of_nodes + np.maximum(start, end)
    _, first_rows = np.unique(pair_key, return_index=True)
    start, end = start[first_rows], end[first_rows]

    road_vectors = (node_embeddings[start] + node_embeddings[end]) / 2
    road_midpoints = (node_coordinates[start] + node_coordinates[end]) / 2

    targets = {}
    for radius in radii_m:
        for kind in place_kinds:
            targets[f"{kind}_{radius}m"] = np.log1p(feature_table[f"nearby_places_{kind}_{radius}m"].to_numpy()[first_rows])
    targets["all_places_1000m"] = np.log1p(sum(feature_table[f"nearby_places_{kind}_1000m"].to_numpy()[first_rows]
                                               for kind in place_kinds))
    return road_vectors, road_midpoints, pd.DataFrame(targets)


road_vectors, road_midpoints, targets = build_physical_road_table(
    feature_table, node_embeddings, node_coordinates, PLACE_KINDS, RADII_M)
print(f"{len(road_vectors):,} physical roads, {road_vectors.shape[1]} Node2Vec dimensions, {targets.shape[1]} targets")

65,252 physical roads, 128 Node2Vec dimensions, 15 targets


## 3. Spatial blocks for a fair test

Neighbouring roads share intersections, so their Node2Vec vectors and their nearby-place counts are both similar. With random folds a model could score well just by
recognising a road's neighbour. Instead, Delaware is cut into a 10 x 10 grid (equal numbers of roads per row and column), and whole grid cells are held out together.

In [4]:
def assign_spatial_folds(road_midpoints, grid_size, number_of_folds, seed):
    """Give every road a fold number so that each grid cell lies entirely in one fold.

    Takes: road midpoints (lat, lon), the grid size, the number of folds and a random seed. Returns an integer array of folds.
    """
    cell_index = []
    for axis in (0, 1):
        edges = np.quantile(road_midpoints[:, axis], np.linspace(0, 1, grid_size + 1)[1:-1])
        cell_index.append(np.searchsorted(edges, road_midpoints[:, axis]))
    block = cell_index[0] * grid_size + cell_index[1]
    fold_of_block = np.random.default_rng(seed).permutation(grid_size * grid_size) % number_of_folds
    return fold_of_block[block]


folds = assign_spatial_folds(road_midpoints, GRID_SIZE, NUMBER_OF_FOLDS, RANDOM_SEED)
print("roads per fold:", np.bincount(folds))

roads per fold: [12641 14617 16169  9804 12021]


## 4. Predict the surroundings from Node2Vec

Held-out R² compares the predictions with simply guessing the training average: 1 is perfect, 0 is no better than the average.
Ridge regression (linear) is run for every target; gradient boosting (non-linear) for the three headline targets. As a control, the same ridge model is fitted
with the Node2Vec vectors shuffled between roads, which should score about 0.

In [5]:
def held_out_r2(features, target, folds, make_model):
    """Spatially blocked R^2: fit on all folds but one, predict the held-out fold, pool the errors over folds.

    Takes: the input matrix, one target vector, the fold of every row, and a function returning a fresh model.
    Returns R^2 = 1 - (held-out squared error) / (squared error of predicting the training mean).
    """
    squared_error, squared_error_of_mean = 0.0, 0.0
    for fold in np.unique(folds):
        train, test = folds != fold, folds == fold
        model = make_model().fit(features[train], target[train])
        squared_error += ((target[test] - model.predict(features[test])) ** 2).sum()
        squared_error_of_mean += ((target[test] - target[train].mean()) ** 2).sum()
    return 1 - squared_error / squared_error_of_mean


def make_ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=RIDGE_ALPHA))


def make_boosting():
    return HistGradientBoostingRegressor(max_iter=300, random_state=RANDOM_SEED)


HEADLINE_TARGETS = ["all_places_1000m", "signal_1000m", "food_1000m"]
shuffled_vectors = road_vectors[np.random.default_rng(RANDOM_SEED).permutation(len(road_vectors))]

rows = []
for name in targets.columns:
    target = targets[name].to_numpy()
    row = {"target": name,
           "share of roads with at least one": float((target > 0).mean()),
           "ridge R2": held_out_r2(road_vectors, target, folds, make_ridge),
           "ridge R2, shuffled control": held_out_r2(shuffled_vectors, target, folds, make_ridge)}
    if name in HEADLINE_TARGETS:
        row["boosting R2"] = held_out_r2(road_vectors, target, folds, make_boosting)
    rows.append(row)
    print(f"{name:20s} ridge {row['ridge R2']:.3f}  control {row['ridge R2, shuffled control']:.3f}"
          + (f"  boosting {row['boosting R2']:.3f}" if "boosting R2" in row else ""))

scores = pd.DataFrame(rows).set_index("target")
display(scores.round(3))

signal_250m          ridge -0.008  control -0.002
food_250m            ridge -0.053  control -0.003
crossing_250m        ridge -0.013  control -0.002
parking_250m         ridge -0.032  control -0.003
bus_stop_250m        ridge -0.055  control -0.003
school_250m          ridge -0.042  control -0.002
bar_pub_250m         ridge -0.045  control -0.002
signal_1000m         ridge 0.012  control -0.003  boosting 0.068
food_1000m           ridge -0.040  control -0.003  boosting -0.006
crossing_1000m       ridge -0.004  control -0.002
parking_1000m        ridge -0.011  control -0.002
bus_stop_1000m       ridge -0.101  control -0.002
school_1000m         ridge -0.059  control -0.003
bar_pub_1000m        ridge -0.102  control -0.002
all_places_1000m     ridge 0.012  control -0.002  boosting 0.063


,share of roads with at least one,ridge R2,"ridge R2, shuffled control",boosting R2
target,,,,
signal_250m,0.239,-0.008,-0.002,NaN
food_250m,0.148,-0.053,-0.003,NaN
crossing_250m,0.321,-0.013,-0.002,NaN
parking_250m,0.287,-0.032,-0.003,NaN
bus_stop_250m,0.028,-0.055,-0.003,NaN
school_250m,0.028,-0.042,-0.002,NaN
bar_pub_250m,0.018,-0.045,-0.002,NaN
signal_1000m,0.656,0.012,-0.003,0.068
food_1000m,0.539,-0.040,-0.003,-0.006


## 5. Save the result

In [6]:
summary = {
    "state": STATE,
    "node2vec_file": NODE2VEC_PATH.name,
    "physical_roads": int(len(road_vectors)),
    "folds": NUMBER_OF_FOLDS, "grid_size": GRID_SIZE, "ridge_alpha": RIDGE_ALPHA, "seed": RANDOM_SEED,
    "target_transform": "log1p(count)",
    "scores": scores.reset_index().to_dict(orient="records"),
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
output_path = RESULTS_DIR / f"node2vec_density_check_{STATE}.json"
output_path.write_text(json.dumps(summary, indent=2))
print("saved", output_path)

saved /content/drive/MyDrive/cs5284_project/results/node2vec_density_check_DE.json


## 6. The same test with random folds

Random folds put roads from every area into both the fitting and the test data, as in the project's temporal evaluation (same roads, later years).
Comparing with section 4 shows whether Node2Vec describes surroundings in general or only identifies locations it has seen.

In [8]:
random_folds = np.random.default_rng(RANDOM_SEED).permutation(len(road_vectors)) % NUMBER_OF_FOLDS
comparison_rows = []
for name in targets.columns:
    target = targets[name].to_numpy()
    row = {"target": name,
           "ridge R2, spatial blocks": scores.loc[name, "ridge R2"],
           "ridge R2, random folds": held_out_r2(road_vectors, target, random_folds, make_ridge)}
    if name in HEADLINE_TARGETS:
        row["boosting R2, spatial blocks"] = scores.loc[name, "boosting R2"]
        row["boosting R2, random folds"] = held_out_r2(road_vectors, target, random_folds, make_boosting)
    comparison_rows.append(row)
    print(f"{name:20s} ridge: spatial {row['ridge R2, spatial blocks']:.3f} -> random {row['ridge R2, random folds']:.3f}"
          + (f" | boosting: spatial {row['boosting R2, spatial blocks']:.3f} -> random {row['boosting R2, random folds']:.3f}"
             if "boosting R2, random folds" in row else ""))

fold_comparison = pd.DataFrame(comparison_rows).set_index("target")
display(fold_comparison.round(3))

summary["fold_comparison"] = fold_comparison.reset_index().to_dict(orient="records")
output_path.write_text(json.dumps(summary, indent=2))
print("updated", output_path)

signal_250m          ridge: spatial -0.008 -> random 0.123
food_250m            ridge: spatial -0.053 -> random 0.070
crossing_250m        ridge: spatial -0.013 -> random 0.128
parking_250m         ridge: spatial -0.032 -> random 0.107
bus_stop_250m        ridge: spatial -0.055 -> random 0.077
school_250m          ridge: spatial -0.042 -> random 0.019
bar_pub_250m         ridge: spatial -0.045 -> random 0.035
signal_1000m         ridge: spatial 0.012 -> random 0.190 | boosting: spatial 0.068 -> random 0.695
food_1000m           ridge: spatial -0.040 -> random 0.136 | boosting: spatial -0.006 -> random 0.669
crossing_1000m       ridge: spatial -0.004 -> random 0.173
parking_1000m        ridge: spatial -0.011 -> random 0.155
bus_stop_1000m       ridge: spatial -0.101 -> random 0.133
school_1000m         ridge: spatial -0.059 -> random 0.120
bar_pub_1000m        ridge: spatial -0.102 -> random 0.116
all_places_1000m     ridge: spatial 0.012 -> random 0.168 | boosting: spatial 0.063 -> ran

,"ridge R2, spatial blocks","ridge R2, random folds","boosting R2, spatial blocks","boosting R2, random folds"
target,,,,
signal_250m,-0.008,0.123,NaN,NaN
food_250m,-0.053,0.070,NaN,NaN
crossing_250m,-0.013,0.128,NaN,NaN
parking_250m,-0.032,0.107,NaN,NaN
bus_stop_250m,-0.055,0.077,NaN,NaN
school_250m,-0.042,0.019,NaN,NaN
bar_pub_250m,-0.045,0.035,NaN,NaN
signal_1000m,0.012,0.190,0.068,0.695
food_1000m,-0.040,0.136,-0.006,0.669


updated /content/drive/MyDrive/cs5284_project/results/node2vec_density_check_DE.json
